## Production Environment

The model was trained and validated with these exact versions. Using different versions may cause errors when loading the pipeline, or silently produce different predictions, because the `.joblib` file is a serialized Python object.

| Library | Version |
|---|---|
| scikit-learn | 1.9.0 |
| xgboost | 3.4.1 |
| pandas | 3.0.3 |
| joblib | 1.5.3 |
| pyodbc | 5.3.0 | SQL Server connection (ODBC) |
| python-dotenv | 1.2.1 | Reads settings from `.env` |

## Model

| Field | Value |
|---|---|
| Problem | Customer churn prediction |
| Algorithm | XGBoost (tuned) |
| Artifact | `customersretention_pipeline.joblib` (preprocessing + model) |
| Configuration | `customersretention_threshold.json` |
| Training date | 2026-10-31 |
| Input features | 19 |

### Decision Threshold

The model flags a customer as **at risk of churning** when the estimated probability is **≥ 0.42**, instead of the default 0.5. The threshold was chosen to prioritize **recall**: it is better to offer a retention incentive to a customer who wasn't going to leave (offer cost: USD 50) than to lose one who was (12 months of revenue).

For this reason, the API uses `predict_proba()` and applies the threshold, instead of `predict()`.

### Test Metrics

| Metric | Value |
|---|---|
| ROC AUC | 0.862 |
| PR AUC | 0.680 |
| Recall | 0.856 |
| Precision | 0.487 |

**Business interpretation:** the model catches 86 out of every 100 customers who actually churn. Of every 100 customers it flags as at risk, 49 were actually going to leave.


In [1]:
# 1. Verify versions

import sklearn, xgboost, pandas, joblib
import sys, numpy, fastapi, pydantic, uvicorn

print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("pandas:", pandas.__version__)
print("joblib:", joblib.__version__)
print("python:", sys.version.split()[0])
print("numpy:", numpy.__version__)
print("fastapi:", fastapi.__version__)
print("pydantic:", pydantic.__version__)
print("uvicorn:", uvicorn.__version__)

scikit-learn: 1.9.0
xgboost: 3.4.1
pandas: 3.0.3
joblib: 1.5.3
python: 3.14.6
numpy: 2.4.6
fastapi: 0.162.0
pydantic: 2.13.4
uvicorn: 0.40.0


In [2]:
#2. Load the file pipeline .joblib
import json

pipeline = joblib.load("customersretention_pipeline.joblib")

with open("customersretention_threshold.json") as f:
    config = json.load(f)

print(pipeline)
print(pipeline.named_steps.keys())

Pipeline(steps=[('prep',
                 Pipeline(steps=[('features',
                                  FunctionTransformer(func=<function engineer_features at 0x0000029B1F5DE610>)),
                                 ('encode',
                                  ColumnTransformer(transformers=[('num',
                                                                   'passthrough',
                                                                   ['tenure',
                                                                    'MonthlyCharges',
                                                                    'TotalCharges',
                                                                    'n_protection']),
                                                                  ('cat',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False),

In [3]:
encoder = pipeline.named_steps["prep"].named_steps["encode"].named_transformers_["cat"]
print(encoder)
print(encoder.handle_unknown)

OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ignore


In [4]:
#3 Get the available Categories

encoder = pipeline.named_steps["prep"].named_steps["encode"].named_transformers_["cat"]

for col, cats in zip(encoder.feature_names_in_, encoder.categories_):
    print(f"{col}: {list(cats)}")

gender: ['Female', 'Male']
SeniorCitizen: ['No', 'Yes']
Partner: ['No', 'Yes']
Dependents: ['No', 'Yes']
PhoneService: ['No', 'Yes']
MultipleLines: ['No', 'Yes']
InternetService: ['DSL', 'Fiber optic', 'No']
OnlineSecurity: ['No', 'Yes']
OnlineBackup: ['No', 'Yes']
DeviceProtection: ['No', 'Yes']
TechSupport: ['No', 'Yes']
StreamingTV: ['No', 'Yes']
StreamingMovies: ['No', 'Yes']
Contract: ['Month-to-month', 'One year', 'Two year']
PaperlessBilling: ['No', 'Yes']
PaymentMethod: ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']


## API Input Contract

The pipeline expects customer data **after** the cleaning step of the training notebook. The API therefore receives data in this format:

| Field | Type | Allowed values |
|---|---|---|
| `tenure` | int | ≥ 0 |
| `MonthlyCharges` | float | > 0 |
| `TotalCharges` | float | ≥ 0 (use `0` for new customers with `tenure = 0`) |
| `SeniorCitizen` | str | `Yes`, `No` (converted from the original 0/1 coding) |
| `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | str | `Yes`, `No`, `No internet service` |
| `MultipleLines` | str | `Yes`, `No`, `No phone service` |
| `InternetService` | str | `DSL`, `Fiber optic`, `No` |
| `Contract` | str | `Month-to-month`, `One year`, `Two year` |
| `PaymentMethod` | str | `Bank transfer (automatic)`, `Credit card (automatic)`, `Electronic check`, `Mailed check` |
| `gender` | str | `Female`, `Male` |
| `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling` | str | `Yes`, `No` |

`customerID` is not used by the model. `"No internet service"` and `"No phone service"` are accepted because `engineer_features` maps them to `"No"` inside the pipeline.

**Future improvement:** move the cleaning step (`TotalCharges` conversion, `SeniorCitizen` mapping) into `features.py`, so the API can accept the original raw format.

In [5]:
# 4 Testing  w/ 3 demo clients using the Decision Threshold

import pandas as pd

clientes = pd.DataFrame([
    {   # Perfil de alto riesgo: cliente nuevo, mes a mes, fibra, sin protección
        "gender": "Female", "SeniorCitizen": "No", "Partner": "No", "Dependents": "No",
        "tenure": 2, "PhoneService": "Yes", "MultipleLines": "No",
        "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
        "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
        "StreamingMovies": "Yes", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
        "PaymentMethod": "Electronic check", "MonthlyCharges": 95.0, "TotalCharges": 190.0,
    },
    {   # Perfil de bajo riesgo: cliente antiguo, contrato de 2 años
        "gender": "Male", "SeniorCitizen": "No", "Partner": "Yes", "Dependents": "Yes",
        "tenure": 60, "PhoneService": "Yes", "MultipleLines": "Yes",
        "InternetService": "DSL", "OnlineSecurity": "Yes", "OnlineBackup": "Yes",
        "DeviceProtection": "Yes", "TechSupport": "Yes", "StreamingTV": "No",
        "StreamingMovies": "No", "Contract": "Two year", "PaperlessBilling": "No",
        "PaymentMethod": "Bank transfer (automatic)", "MonthlyCharges": 70.0, "TotalCharges": 4200.0,
    },
    {   # Cliente sin internet: prueba que engineer_features convierta "No internet service"
        "gender": "Female", "SeniorCitizen": "Yes", "Partner": "No", "Dependents": "No",
        "tenure": 15, "PhoneService": "Yes", "MultipleLines": "No",
        "InternetService": "No", "OnlineSecurity": "No internet service",
        "OnlineBackup": "No internet service", "DeviceProtection": "No internet service",
        "TechSupport": "No internet service", "StreamingTV": "No internet service",
        "StreamingMovies": "No internet service", "Contract": "Month-to-month",
        "PaperlessBilling": "No", "PaymentMethod": "Mailed check",
        "MonthlyCharges": 20.0, "TotalCharges": 300.0,
    },
])

# Mismo orden de columnas que en el entrenamiento
clientes = clientes[config["input_features"]]

proba = pipeline.predict_proba(clientes)[:, 1]
pred_umbral = (proba >= config["threshold"]).astype(int)
pred_default = pipeline.predict(clientes)

resultado = pd.DataFrame({
    "perfil": ["Alto riesgo", "Bajo riesgo", "Sin internet"],
    "probabilidad": proba.round(3),
    "contactar (umbral 0.42)": pred_umbral,
    "predict() (umbral 0.5)": pred_default,
})
resultado

,perfil,probabilidad,contactar (umbral 0.42),predict() (umbral 0.5)
0,Alto riesgo,0.928,1,1
1,Bajo riesgo,0.026,0,0
2,Sin internet,0.197,0,0


In [6]:
malo = clientes.iloc[[0]].copy()
malo["Contract"] = "monthly"          # fake value

print("Original:     ", pipeline.predict_proba(clientes.iloc[[0]])[0, 1].round(3))
print("Con 'monthly':", pipeline.predict_proba(malo)[0, 1].round(3))

Original:      0.928
Con 'monthly': 0.787


In [7]:
# pip install "fastapi[standard]"

In [8]:
import requests

API_URL = "http://127.0.0.1:8000"

# 1. Are you alive?
r = requests.get(f"{API_URL}/health")
print(r.status_code, r.json())

200 {'status': 'ok'}


In [9]:
# 4. Testing with 3 demo customers using the decision threshold
import pandas as pd

high_risk = {   # New customer, month-to-month, fiber, no protection
    "gender": "Female", "SeniorCitizen": "No", "Partner": "No", "Dependents": "No",
    "tenure": 2, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "Yes",
    "StreamingMovies": "Yes", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check", "MonthlyCharges": 95.0, "TotalCharges": 190.0,
}
low_risk = {    # Long-tenure customer, two-year contract
    "gender": "Male", "SeniorCitizen": "No", "Partner": "Yes", "Dependents": "Yes",
    "tenure": 60, "PhoneService": "Yes", "MultipleLines": "Yes",
    "InternetService": "DSL", "OnlineSecurity": "Yes", "OnlineBackup": "Yes",
    "DeviceProtection": "Yes", "TechSupport": "Yes", "StreamingTV": "No",
    "StreamingMovies": "No", "Contract": "Two year", "PaperlessBilling": "No",
    "PaymentMethod": "Bank transfer (automatic)", "MonthlyCharges": 70.0, "TotalCharges": 4200.0,
}
no_internet = { # Checks that engineer_features maps "No internet service" to "No"
    "gender": "Female", "SeniorCitizen": "Yes", "Partner": "No", "Dependents": "No",
    "tenure": 15, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "No", "OnlineSecurity": "No internet service",
    "OnlineBackup": "No internet service", "DeviceProtection": "No internet service",
    "TechSupport": "No internet service", "StreamingTV": "No internet service",
    "StreamingMovies": "No internet service", "Contract": "Month-to-month",
    "PaperlessBilling": "No", "PaymentMethod": "Mailed check",
    "MonthlyCharges": 20.0, "TotalCharges": 300.0,
}

demo_customers = {"High risk": high_risk, "Low risk": low_risk, "No internet": no_internet}

# Same column order as in training
X_demo = pd.DataFrame(list(demo_customers.values()))[config["input_features"]]

proba = pipeline.predict_proba(X_demo)[:, 1]

results = pd.DataFrame({
    "profile": list(demo_customers.keys()),
    "churn_probability": proba.round(3),
    "contact (threshold 0.42)": (proba >= config["threshold"]).astype(int),
    "predict() (threshold 0.5)": pipeline.predict(X_demo),
})
results

,profile,churn_probability,contact (threshold 0.42),predict() (threshold 0.5)
0,High risk,0.928,1,1
1,Low risk,0.026,0,0
2,No internet,0.197,0,0


In [10]:
# 5. Testing the API with requests (the API must be running)
import requests

API_URL = "http://127.0.0.1:8000"

r = requests.get(f"{API_URL}/health")
print("health:", r.status_code, r.json())

for name, customer in demo_customers.items():
    r = requests.post(f"{API_URL}/predict", json=customer)
    print(f"{name}:", r.status_code, r.json())

health: 200 {'status': 'ok'}
High risk: 200 {'customerID': None, 'churn_probability': 0.928, 'contact': True, 'threshold': 0.42}
Low risk: 200 {'customerID': None, 'churn_probability': 0.026, 'contact': False, 'threshold': 0.42}
No internet: 200 {'customerID': None, 'churn_probability': 0.197, 'contact': False, 'threshold': 0.42}


## Input validation: invalid requests must be rejected

The pipeline's `OneHotEncoder` uses `handle_unknown='ignore'`: an unknown category does not raise an error, it silently produces a wrong prediction (e.g. `Contract = "monthly"` changed the probability from 0.928 to 0.787). For this reason, the API validates every input with Pydantic **before** it reaches the model.

Each case below modifies one field of a valid customer and must return **`422 Unprocessable Content`**:

| Case | Validation rule |
|---|---|
| Invalid category | `Literal`: only the categories seen in training are allowed |
| Wrong type | `tenure` must be an integer |
| Negative `tenure` | `Field(ge=0)` |
| Zero `MonthlyCharges` | `Field(gt=0)`: the training data has no customers with 0 charges |
| Lowercase value | Categories are case-sensitive |
| Missing field | All 19 features are required |
| Extra field | `extra="forbid"`: unknown fields are rejected |

The `assert` makes this cell an automated test: if a future change breaks validation, the cell fails.

In [11]:
# 6. Invalid inputs must be rejected with 422
invalid_cases = {
    "Invalid category (Contract='monthly')": {**high_risk, "Contract": "monthly"},
    "Wrong type (tenure='two')":             {**high_risk, "tenure": "two"},
    "Negative tenure":                       {**high_risk, "tenure": -5},
    "Lowercase value (gender='female')":     {**high_risk, "gender": "female"},
    "Missing field (no Contract)":           {k: v for k, v in high_risk.items() if k != "Contract"},
    "Unknown extra field":                   {**high_risk, "discount": 10},
}

for name, payload in invalid_cases.items():
    r = requests.post(f"{API_URL}/predict", json=payload)
    errors = r.json().get("detail", [])
    msgs = "; ".join(f"{e['loc'][-1]}: {e['msg']}" for e in errors)
    print(f"{r.status_code} | {name}\n      {msgs}\n")
    assert r.status_code == 422, f"Expected 422 for: {name}"

print("All invalid inputs were rejected ✅")

422 | Invalid category (Contract='monthly')
      Contract: Input should be 'Month-to-month', 'One year' or 'Two year'

422 | Wrong type (tenure='two')
      tenure: Input should be a valid integer, unable to parse string as an integer

422 | Negative tenure
      tenure: Input should be greater than or equal to 0

422 | Lowercase value (gender='female')
      gender: Input should be 'Female' or 'Male'

422 | Missing field (no Contract)
      Contract: Field required

422 | Unknown extra field
      discount: Extra inputs are not permitted

All invalid inputs were rejected ✅


## Design decisions: optional `customerID` and zero `MonthlyCharges`

- **`customerID`** is optional. It is **not** passed to the model (it is an identifier, not a feature) and is returned in the response, so the calling system knows which customer each prediction belongs to. The probability is identical with or without it.
- **`MonthlyCharges = 0`** is accepted to support real cases such as promotional months. The training data has no customers below ~18, so predictions in that range are less reliable; this is documented in the API schema.
- Any **other** unknown field is still rejected (`extra="forbid"`).

In [12]:
# 7. Optional customerID and zero MonthlyCharges are accepted
r = requests.post(f"{API_URL}/predict", json={**high_risk, "customerID": "1234-ABCD"})
print("with customerID:", r.status_code, r.json())
assert r.status_code == 200 and r.json()["customerID"] == "1234-ABCD"

r = requests.post(f"{API_URL}/predict", json=high_risk)
print("without customerID:", r.status_code, r.json())
assert r.status_code == 200 and r.json()["customerID"] is None

r = requests.post(f"{API_URL}/predict", json={**high_risk, "MonthlyCharges": 0})
print("zero MonthlyCharges:", r.status_code, r.json())
assert r.status_code == 200

print("All new valid cases accepted ✅")

with customerID: 200 {'customerID': '1234-ABCD', 'churn_probability': 0.928, 'contact': True, 'threshold': 0.42}
without customerID: 200 {'customerID': None, 'churn_probability': 0.928, 'contact': True, 'threshold': 0.42}
zero MonthlyCharges: 200 {'customerID': None, 'churn_probability': 0.879, 'contact': True, 'threshold': 0.42}
All new valid cases accepted ✅


## Running with Docker

**Requirements:** Docker 29+ (Docker Desktop on Windows/Mac).

```bash
# Build the image
docker build -f Dockerfile_customersretention -t customersretention-api .

# Run the container in the background
docker run -d --name churn-api -p 8000:8000 customersretention-api
```

The API is available at http://127.0.0.1:8000/docs.

| Command | Purpose |
|---|---|
| `docker ps` | Check that the container is running |
| `docker logs churn-api` | View the API logs |
| `docker stop churn-api` | Stop the container |
| `docker start churn-api` | Start it again |
| `docker rm churn-api` | Remove the container |

The container reproduces the training environment exactly (Python 3.14, scikit-learn 1.9.0, xgboost 3.4.1, pandas 3.0.3). The test sections of `despliegue_produccion.ipynb` return identical results locally and in Docker.

## Database endpoints

The API reads customers from SQL Server through stored procedures, using a service account with EXECUTE-only permissions.

| Case | Endpoint | Expected |
|---|---|---|
| Existing customer | `GET /customers/7590-VHVEG/prediction` | `200`, prediction with the customerID |
| Unknown customer | `GET /customers/XXXX-XXXXX/prediction` | `404` |
| Batch scoring | `POST /batch/score` | `200`, 5174 active customers scored, 0 invalid |

> Requires the API (`fastapi dev main_customersretention.py`) and the `sqlserver` container to be running. Each batch run adds a new set of rows to `ml.ChurnPredictions`.

In [13]:
# 8. Database endpoints
r = requests.get(f"{API_URL}/customers/7590-VHVEG/prediction")
print("existing customer:", r.status_code, r.json())
assert r.status_code == 200 and r.json()["customerID"] == "7590-VHVEG"

r = requests.get(f"{API_URL}/customers/XXXX-XXXXX/prediction")
print("unknown customer:", r.status_code, r.json())
assert r.status_code == 404

r = requests.post(f"{API_URL}/batch/score")
summary = r.json()
print("batch:", r.status_code, summary)
assert r.status_code == 200
assert summary["scored"] == 5174 and summary["invalid"] == 0

print("All database endpoint tests passed ✅")

existing customer: 200 {'customerID': '7590-VHVEG', 'churn_probability': 0.865, 'contact': True, 'threshold': 0.42}
unknown customer: 404 {'detail': "Customer 'XXXX-XXXXX' not found"}
batch: 200 {'batch_id': '6c4b8a30-66ad-4c48-b882-e56340be6ccd', 'scored': 5174, 'to_contact': 1678, 'invalid': 0, 'invalid_ids': [], 'threshold': 0.42, 'seconds': 0.21}
All database endpoint tests passed ✅


## Database Integration (SQL Server)

The API reads customers directly from SQL Server, so callers only need a customer ID. Data cleaning moved from pandas to SQL: the database stores raw CRM-style data and a view returns it in the model's input format.

```
SQL Server (Docker container)                    API (FastAPI)
┌──────────────────────────────────┐            ┌────────────────────────────────────┐
│ dbo.Customers         (raw data) │            │                                    │
│ dbo.vw_CustomerFeatures (cleaning)│            │                                    │
│ dbo.usp_GetCustomerFeatures   ───┼───────────►│ GET /customers/{id}/prediction     │
│ dbo.usp_GetCustomersForScoring ──┼───────────►│ POST /batch/score                  │
│ ml.ChurnPredictions          ◄───┼────────────┤   (stores every prediction)        │
│ ml.usp_GetRetentionList          │            │                                    │
└──────────────────────────────────┘            └────────────────────────────────────┘
```

| Cleaning rule | Before (pandas) | Now (SQL view) |
|---|---|---|
| `SeniorCitizen` 0/1 → "No"/"Yes" | `.map({1: "Yes", 0: "No"})` | `CASE SeniorCitizen WHEN 1 THEN 'Yes' ELSE 'No' END` |
| `TotalCharges` missing → 0 | `pd.to_numeric(...).fillna(0)` | `ISNULL(TotalCharges, 0)` |

The rules must match the cleaning used for training; otherwise the model receives data different from what it learned (training-serving skew).

## Database Setup

SQL Server 2022 (Developer Edition) runs in a Docker container. Data is stored in a named volume, so it survives container removal.

```bash
docker run -d --name sqlserver -e "ACCEPT_EULA=Y" -e "MSSQL_SA_PASSWORD=<password>" \
  -p 1433:1433 -v sqlserver-data:/var/opt/mssql mcr.microsoft.com/mssql/server:2022-latest
```

Run the scripts in `sql/` in order, then load the data with `database_setup.ipynb` (the Telco CSV must be in `data/`):

| Script | Creates |
|---|---|
| `01_create_database.sql` | `ChurnDB` |
| `02_create_tables.sql` | `dbo.Customers` (raw CRM-style data) |
| `03_views_and_procedures.sql` | Cleaning view and the two read procedures |
| `04_api_user.sql` | Service account `api_user_ChurnDB` |
| `05_predictions.sql` | Schema `ml`, table `ml.ChurnPredictions`, retention list procedure |
| `06_analysis_queries.sql` | Analysis queries (not part of setup) |

All setup scripts are re-runnable (`IF ... IS NULL`, `CREATE OR ALTER`).

**Security.** The API connects with `api_user_ChurnDB`, which can only `EXECUTE` the two read procedures and `INSERT` into `ml.ChurnPredictions`. It cannot read tables directly (verified: `SELECT` on `dbo.Customers` is denied) nor access other databases. Connection settings are read from `.env` (excluded from git); see `.env.example`.

## Database Endpoints

| Endpoint | What it does | Responses |
|---|---|---|
| `GET /customers/{customer_id}/prediction` | Reads one customer through `usp_GetCustomerFeatures`, validates it and scores it | `200` prediction · `404` unknown customer · `500` database data failed validation · `503` database unavailable |
| `POST /batch/score` | Scores all active customers (`Churn = 'No'`) and stores the results with a batch ID, model name and training date | `200` summary · `503` database unavailable |

Data read from the database goes through the same Pydantic validation as data sent by callers. In a batch, invalid rows are reported and skipped instead of stopping the run.

**Results:** 5,174 active customers scored in 0.33 s; 1,678 flagged for contact. Two runs gave identical results (reproducible).

**Retention list.** `ml.usp_GetRetentionList` returns every customer of the latest run, ranked by risk, with the action (`Contact` / `No action`) and the expected revenue at risk over 12 months (probability × monthly charge × 12). Ranking by revenue at risk instead of probability prioritises different customers, which matters when the retention budget is limited.

> These customers belong to the dataset used for training, so the scores demonstrate the pipeline end to end; they are not an evaluation of the model.

## From Local to Production

The local SQL Server container stands in for a company database. In a real environment the API code stays the same; what changes:

| Topic | This project | Production |
|---|---|---|
| Database | Created locally in Docker | Existing server, procedures maintained by the DBA / data team |
| Connection | `.env` file | Same variables, provided by the deployment platform or a secrets manager |
| API account | `api_user_ChurnDB` | Service account with the same least-privilege grants |
| Predictions | `ml` schema in the same database | Dedicated schema agreed with the DBA |
| Batch trigger | `POST /batch/score` called manually | Scheduled job (e.g. SQL Server Agent, scheduler) calling the same endpoint |
| Data | Public Telco dataset | Development or test copy with personal data masked; never tested against production |

Two common failure points: existing procedures returning different column names or codes than the model expects (Pydantic validation catches it), and cleaning rules diverging between training and serving (training data should be extracted with the same procedure).

## Changelog

| Version | Date | Changes |
|---|---|---|
| 1.0.0 | 2026-11-04 | API with `/health`, `/model-info`, `/predict` (threshold 0.42); strict input validation |
| 1.1.0 | 2026-11-06 | Optional `customerID` returned in the response; `MonthlyCharges = 0` accepted; Docker image |
| 1.2.0 | 2026-11-07 | SQL Server integration: `GET /customers/{customer_id}/prediction`; settings from `.env`; least-privilege service account |
| 1.3.0 | 2026-11-07 | Batch scoring: `POST /batch/score` stores predictions in `ml.ChurnPredictions` with traceability. Docker Compose runs the API and SQL Server together |

In [14]:
from importlib.metadata import version
print("pyodbc:", version("pyodbc"))
print("python-dotenv:", version("python-dotenv"))

pyodbc: 5.3.0
python-dotenv: 1.2.1


## Running the Full System (Docker Compose)

The API and SQL Server run together with one command. Compose creates a private network where the API reaches the database by service name (`sqlserver,1433`), and the API starts only after SQL Server passes its health check.

```bash
cp .env.example .env              # fill in the passwords
docker volume create sqlserver-data
docker compose up -d
```

On a fresh volume, create the database with the scripts in `sql/` and load the data with `database_setup.ipynb` before using the database endpoints.

| Service | Image | Port | Notes |
|---|---|---|---|
| `sqlserver` | `mssql/server:2022-latest` | 1433 | Data in the `sqlserver-data` volume; health check with `SELECT 1` |
| `api` | Built from `Dockerfile_customersretention` | 8000 | Includes the Microsoft ODBC Driver 18; receives only the `DB_*` settings |

No credentials are stored in `docker-compose.yml` or in the image: they are read from `.env` at startup.